# Why noise is the enemy of quantum chemistry: a demo with the simplest molecule
**Qiskit Fall Fest demo**

Medicines work by molecules fitting together. If we could compute exactly how molecules behave, we could design medicines faster. Quantum computers may help one day, but today's quantum chips make small mistakes ("noise").

In this notebook we:
1. Use a quantum method to work out the energy of **hydrogen gas (H₂)**, the simplest molecule there is.
2. Check that answer against the exact answer from a normal computer.
3. Add 1% and 3% chip errors and see how far off the answer gets.

> ⚠️ **This is only a teaching example.** H₂ is not a medicine, and nothing here treats typhoid or any other disease. We use the simplest possible molecule to show the main problem.

Total run time: about 1–2 minutes.

## Step 0: Install the tools
This installs the free software we need: Qiskit (IBM's quantum toolkit), a quantum *simulator* that runs on this computer, and a chemistry package that describes the molecule. Run it once; it takes about 30–60 seconds.

In [ ]:
!pip install -q qiskit qiskit-aer matplotlib numpy scipy qiskit-nature qiskit-algorithms pyscf

## Step 1: Load the tools and start a timer
Here we load the software we just installed so Python can use it. Nothing is calculated yet.

In [ ]:
import time, warnings
warnings.filterwarnings("ignore")
t0 = time.time()
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from qiskit import transpile
from qiskit_aer.primitives import EstimatorV2 as AerEstimator
from qiskit_aer.noise import NoiseModel, depolarizing_error
from qiskit_algorithms.optimizers import SPSA
from qiskit_algorithms.utils import algorithm_globals
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.mappers import ParityMapper
from qiskit_nature.second_q.circuit.library import HartreeFock, UCCSD
np.random.seed(1); algorithm_globals.random_seed = 1
MILLI = 1000   # we show errors in thousandths of an energy unit
print("Tools loaded.")

## Step 2: Describe the molecule and turn it into a quantum program
We tell the computer: two hydrogen atoms, a set distance apart. The chemistry package turns this into a puzzle a quantum computer can work on, using **2 qubits** (a qubit is the quantum version of a bit).

We also build the quantum program itself, a short list of steps with "dials" we can turn. Turning the dials changes the answer the program gives.

The helper below does all this for any distance between the two atoms. It also works out the **exact answer** the normal way, which only works because the molecule is tiny.

In [ ]:
def build(distance):
    problem = PySCFDriver(atom=f"H 0 0 0; H 0 0 {distance}", basis="sto3g").run()
    mapper = ParityMapper(num_particles=problem.num_particles)
    puzzle = mapper.map(problem.hamiltonian.second_q_op())
    push_apart = problem.nuclear_repulsion_energy          # the two nuclei repel each other
    exact = np.linalg.eigvalsh(puzzle.to_matrix()).min() + push_apart
    start = HartreeFock(problem.num_spatial_orbitals, problem.num_particles, mapper)
    program = UCCSD(problem.num_spatial_orbitals, problem.num_particles, mapper, initial_state=start)
    program = transpile(program, basis_gates=["cx", "rz", "sx", "x"], optimization_level=1)
    return puzzle, push_apart, exact, program

puzzle, push_apart, exact, program = build(0.735)
print("Qubits used:", puzzle.num_qubits)
print("Dials to tune:", program.num_parameters)
print("Two-qubit steps in the program:", program.count_ops().get("cx", 0))
print(f"Exact energy at normal bond length (0.735 Å): {exact:.6f} Hartree")

## Step 3: Let the quantum method find the energy (no noise yet)
This method is called **VQE**. It works like a guessing game:
1. The quantum computer (here, a perfect simulator) runs the program and reports an energy.
2. A normal computer turns the dials to try to get a lower energy.
3. Repeat until the energy stops going down.

Nature always settles into the lowest energy, so the lowest number we find is our answer. Then we compare it with the exact answer.

In [ ]:
perfect = AerEstimator(options={"default_precision": 0})
def energy(estimator, program, puzzle, push_apart, dials):
    return float(estimator.run([(program, puzzle, dials)]).result()[0].data.evs) + push_apart

def vqe(distance):
    puzzle, push_apart, exact, program = build(distance)
    f = lambda x: energy(perfect, program, puzzle, push_apart, x)
    start_guess = f(np.zeros(program.num_parameters))   # textbook first guess
    r = minimize(f, np.zeros(program.num_parameters), method="COBYLA", options={"maxiter": 200})
    return dict(d=distance, exact=exact, vqe=r.fun, first_guess=start_guess)

best = vqe(0.735)
ideal_err = (best["vqe"] - best["exact"]) * MILLI
print(f"Quantum method (VQE): {best['vqe']:.6f} Hartree")
print(f"Exact answer:         {best['exact']:.6f} Hartree")
print(f"Error:                {abs(ideal_err):.4f} milli-Hartree")
print("Chemists call an error below 1.6 milli-Hartree 'chemical accuracy'.",
      "PASSED" if abs(ideal_err) < 1.6 else "MISSED")

## Step 4: Repeat for different distances between the atoms
If we pull the two atoms apart or push them together, the energy changes. The lowest point of this curve is where the molecule is most comfortable, which is its real size. We check that the quantum method matches the exact answer at every distance.

In [ ]:
distances = [0.4, 0.55, 0.735, 0.9, 1.2, 1.6, 2.0, 2.5]
curve = [vqe(d) for d in distances]
for c in curve:
    print(f"distance {c['d']:.3f} Å   exact {c['exact']:.5f}   VQE {c['vqe']:.5f}   error {abs(c['vqe']-c['exact'])*MILLI:.4f} mHa")

## Step 5: Now add the chip's mistakes (noise)
Real quantum chips are not perfect. Each step has a small chance of going wrong, and the answer is read out by measuring many times (like a poll with limited answers), so there's also random wobble.

We try three settings:
- **Measuring only**: a perfect chip, measured 8,192 times
- **1% noise**: each two-qubit step goes wrong 1% of the time
- **3% noise**: each two-qubit step goes wrong 3% of the time

For each one we run the same guessing game again (using an optimizer that copes with wobbly answers), then measure the final energy. This step takes the longest, about a minute.

In [ ]:
puzzle, push_apart, exact, program = build(0.735)
results = {}
for label, p in [("Measuring only", 0), ("1% noise", 0.01), ("3% noise", 0.03)]:
    opts = {"default_precision": 1/np.sqrt(8192)}
    if p:
        nm = NoiseModel()
        nm.add_all_qubit_quantum_error(depolarizing_error(p, 2), ["cx"])
        nm.add_all_qubit_quantum_error(depolarizing_error(p/10, 1), ["sx", "x"])
        opts["backend_options"] = {"noise_model": nm}
    chip = AerEstimator(options=opts)
    r = SPSA(maxiter=150).minimize(lambda x: energy(chip, program, puzzle, push_apart, x), np.zeros(program.num_parameters))
    final = np.mean([energy(chip, program, puzzle, push_apart, r.x) for _ in range(20)])
    results[label] = (final - exact) * MILLI
    print(f"{label:15s} energy {final:.5f}  error {abs(results[label]):7.2f} mHa  ->",
          "within chemical accuracy" if abs(results[label]) < 1.6 else "TOO INACCURATE for chemistry")

## Step 6: The picture
**Left:** the energy curve. The quantum method (dots) lands right on the exact answer (line). The dashed line is the textbook first guess, which is clearly worse.

**Right:** how big the error gets as noise is added (note the scale goes up 10× per gridline). The grey line is chemical accuracy. Anything above it is too inaccurate to be useful for chemistry.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
d = [c["d"] for c in curve]
ax[0].plot(d, [c["exact"] for c in curve], "k-", label="Exact answer")
ax[0].plot(d, [c["vqe"] for c in curve], "o", label="Quantum method (VQE, perfect simulator)")
ax[0].plot(d, [c["first_guess"] for c in curve], "--", label="Textbook first guess")
ax[0].set_xlabel("Distance between the two atoms (Å)"); ax[0].set_ylabel("Energy (Hartree)")
ax[0].set_title("H₂ energy vs. distance"); ax[0].legend()
labels = ["Perfect\nchip"] + [k.replace(" ", "\n") for k in results]
vals = [max(abs(ideal_err), 1e-4)] + [abs(v) for v in results.values()]
ax[1].bar(labels, vals, color=["tab:green", "tab:blue", "tab:orange", "tab:red"])
ax[1].axhline(1.6, ls="--", color="gray"); ax[1].text(-0.4, 2.0, "chemical accuracy (1.6)", color="gray")
ax[1].set_yscale("log"); ax[1].set_ylabel("Error vs exact (milli-Hartree)"); ax[1].set_title("Error at normal bond length")
plt.suptitle("Teaching example: VQE on hydrogen (H₂). Not a drug, not related to any medicine.")
plt.tight_layout(); plt.show()
print(f"Total run time: {time.time()-t0:.0f} s")

## What we learned
- On a **perfect** (simulated) quantum computer, the method gets the right answer for H₂.
- Just **1% noise** makes the error over 10× too big for chemistry, and **3%** makes it worse still.
- Real medicine molecules need hundreds or thousands of qubits and much longer programs, so the mistakes add up even more.

That is why reducing noise and correcting errors is the key problem to solve before quantum computers can help discover drugs. Again, this was a teaching example with hydrogen, not a medicine.